# Lending Club - Credit Risk Analysis

**Business question:** Who defaults on their loans, can we spot risky borrowers before lending, and how much money is at risk?

**Data:** Real Lending Club loans (Kaggle), 2,260,668 loans x 145 columns.

**Workflow:** Ask -> Prepare -> Process (clean) -> Analyze -> Share -> Act

## Step 1 - Peek at the structure
load a 1.2 GB file. Load a tiny slice first just to see the columns.

In [40]:
import pandas as pd
df = pd.read_csv('loan.csv', nrows = 1000)
print(df.shape)
print(df.columns.tolist())

(1000, 145)
['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'open_acc_6m', 'open_act_il', 'open_il_1

## Step 2 - Select the credit-risk columns (one-time heavy cell)
Of 145 columns we keep ~22 that a lender actually knows AT the moment of lending.

**Data leakage trap:** columns like `total_pymnt`, `recoveries`, `out_prncp` describe what happened AFTER the loan played out. Using them to 'predict' default is cheating. So we exclude them.

This cell reads the full 1.2 GB file once (~1 min) and saves a slim file. Run it once; after that we just load `loan_slim.csv`.

In [41]:
keep = [
  # the loan itself
  "loan_amnt", "term", "int_rate", "installment", "grade", "sub_grade", "purpose",
  # the borrower
  "annual_inc", "emp_length", "home_ownership", "verification_status", "addr_state",
  # their credit health
  "dti", "delinq_2yrs", "open_acc", "pub_rec", "revol_util", "total_acc",
  "pub_rec_bankruptcies", "earliest_cr_line",
  # timing
  "issue_d",
  # the TARGET
  "loan_status",
]

df = pd.read_csv('loan.csv', usecols= keep)
print('Shape:', df.shape)

df.to_csv('loan_slim.csv', index=False)
print('Saved to loan_slim.csv')



Shape: (2260668, 22)
Saved to loan_slim.csv


## Step 3 - Profile the data
What types did each column get? What is missing? What does the target column contain?

In [42]:
df = pd.read_csv('loan_slim.csv')
df.info()

print('---Missing Valus per Column---')
print(df.isna().sum())

print('\n---Loan_status Categories---')
print(df['loan_status'].value_counts())

<class 'pandas.DataFrame'>
RangeIndex: 2260668 entries, 0 to 2260667
Data columns (total 22 columns):
 #   Column                Dtype  
---  ------                -----  
 0   loan_amnt             int64  
 1   term                  str    
 2   int_rate              float64
 3   installment           float64
 4   grade                 str    
 5   sub_grade             str    
 6   emp_length            str    
 7   home_ownership        str    
 8   annual_inc            float64
 9   verification_status   str    
 10  issue_d               str    
 11  loan_status           str    
 12  purpose               str    
 13  addr_state            str    
 14  dti                   float64
 15  delinq_2yrs           float64
 16  earliest_cr_line      str    
 17  open_acc              float64
 18  pub_rec               float64
 19  revol_util            float64
 20  total_acc             float64
 21  pub_rec_bankruptcies  float64
dtypes: float64(10), int64(1), str(11)
memory usage: 552.4

In [43]:
keep = [
  # the loan itself
  "loan_amnt", "term", "int_rate", "installment", "grade", "sub_grade", "purpose",
  # the borrower
  "annual_inc", "emp_length", "home_ownership", "verification_status", "addr_state",
  # their credit health
  "dti", "delinq_2yrs", "open_acc", "pub_rec", "revol_util", "total_acc",
  "pub_rec_bankruptcies", "earliest_cr_line",
  # timing
  "issue_d",
  # the TARGET
  "loan_status",
]

df = pd.read_csv("loan.csv", usecols=keep)   # big file, one last time (~1 min)
print("Shape:", df.shape)
df.to_csv("loan_slim.csv", index=False)
print("Saved loan_slim.csv ✅ (now with int_rate)")

Shape: (2260668, 22)
Saved loan_slim.csv ✅ (now with int_rate)


In [44]:
df = pd.read_csv('loan_slim.csv')
df.info()
print('---Missing Valus per Column---')
print(df.isna().sum())
print('\n---Loan_status Categories---')
print(df['loan_status'].value_counts())

<class 'pandas.DataFrame'>
RangeIndex: 2260668 entries, 0 to 2260667
Data columns (total 22 columns):
 #   Column                Dtype  
---  ------                -----  
 0   loan_amnt             int64  
 1   term                  str    
 2   int_rate              float64
 3   installment           float64
 4   grade                 str    
 5   sub_grade             str    
 6   emp_length            str    
 7   home_ownership        str    
 8   annual_inc            float64
 9   verification_status   str    
 10  issue_d               str    
 11  loan_status           str    
 12  purpose               str    
 13  addr_state            str    
 14  dti                   float64
 15  delinq_2yrs           float64
 16  earliest_cr_line      str    
 17  open_acc              float64
 18  pub_rec               float64
 19  revol_util            float64
 20  total_acc             float64
 21  pub_rec_bankruptcies  float64
dtypes: float64(10), int64(1), str(11)
memory usage: 552.4

Step 4
Define The Target

In [45]:
df = pd.read_csv('loan_slim.csv')
df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off', 'Default'])].copy()
df.shape
df['defaulted'] = df['loan_status'].isin(['Charged Off', 'Default']).astype(int)
print(df.shape)
print(df['defaulted'].value_counts())
print(df['defaulted'].mean())


(1303638, 23)
defaulted
0    1041952
1     261686
Name: count, dtype: int64
0.20073517341470562


In [46]:
df.shape
df['defaulted'].mean()

np.float64(0.20073517341470562)

Step 5: fix the 4 text columns so we can do math / dates with them

In [47]:
df['term'] = df['term'].str.extract(r'(\d+)', expand=False).astype(int)
df['emp_length'] = df['emp_length'].str.extract(r'(\d+)', expand=False).astype(float)
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y')
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y')

print(df[['term', 'emp_length', 'issue_d', 'earliest_cr_line']].dtypes)
print(df[['term', 'emp_length', 'issue_d', 'earliest_cr_line']].head())

term                         int64
emp_length                 float64
issue_d             datetime64[us]
earliest_cr_line    datetime64[us]
dtype: object
     term  emp_length    issue_d earliest_cr_line
100    36        5.00 2018-12-01       2012-01-01
152    60        1.00 2018-12-01       2009-06-01
170    36       10.00 2018-12-01       1999-02-01
186    36       10.00 2018-12-01       2003-12-01
215    36        3.00 2018-12-01       1997-10-01


In [48]:
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%Y')
df[['term', 'emp_length', 'issue_d', 'earliest_cr_line']].dtypes

term                         int64
emp_length                 float64
issue_d             datetime64[us]
earliest_cr_line    datetime64[us]
dtype: object

Step 6 MIssing Values, 

In [49]:
df.isnull().sum().sort_values(ascending=False)

emp_length              75457
revol_util                810
pub_rec_bankruptcies      697
dti                       312
int_rate                    0
term                        0
loan_amnt                   0
sub_grade                   0
grade                       0
installment                 0
home_ownership              0
issue_d                     0
loan_status                 0
verification_status         0
annual_inc                  0
addr_state                  0
purpose                     0
delinq_2yrs                 0
earliest_cr_line            0
pub_rec                     0
open_acc                    0
total_acc                   0
defaulted                   0
dtype: int64

In [50]:
raw = pd.read_csv('loan_slim.csv', usecols=['emp_length'])
raw['emp_length'].value_counts(dropna=False).head(15)

emp_length
10+ years    748005
2 years      203677
< 1 year     189988
3 years      180753
1 year       148403
NaN          146907
5 years      139698
4 years      136605
6 years      102628
7 years       92695
8 years       91914
9 years       79395
Name: count, dtype: int64

In [51]:
df['emp_length'] = raw['emp_length'].str.extract(r'(\d+)', expand=False).astype(float)

print(df['emp_length'].isnull().sum())
df['emp_length'].value_counts(dropna=False).head()

75457


emp_length
10.00    428553
1.00     190230
2.00     117825
3.00     104204
5.00      81623
Name: count, dtype: int64

In [52]:
median_emp = df['emp_length'].median()
df['emp_length'] = df['emp_length'].fillna(median_emp)
print('Filled with: ', median_emp)
print('Missing error: ', df['emp_length'].isnull().sum())

Filled with:  6.0
Missing error:  0


In [53]:
before = len(df)
df =df.dropna(subset=['revol_util', 'dti', 'pub_rec_bankruptcies'])
after = len(df)

print('Dropped: ', before - after, 'rows')
print('Rows Left: ', after)
print("Total Missing now: ", df.isnull().sum().sum())

Dropped:  1819 rows
Rows Left:  1301819
Total Missing now:  0


In [54]:
df.describe()


,loan_amnt,term,int_rate,installment,emp_length,annual_inc,issue_d,dti,delinq_2yrs,earliest_cr_line,open_acc,pub_rec,revol_util,total_acc,pub_rec_bankruptcies,defaulted
count,"1,301,819.00","1,301,819.00","1,301,819.00","1,301,819.00","1,301,819.00","1,301,819.00",1301819,"1,301,819.00","1,301,819.00",1301819,"1,301,819.00","1,301,819.00","1,301,819.00","1,301,819.00","1,301,819.00","1,301,819.00"
mean,"14,420.23",41.80,13.26,438.18,6.05,"76,167.78",2015-05-21 06:53:31.065747,18.27,0.32,1999-02-17 02:53:22.692233,11.60,0.22,51.92,25.03,0.13,0.20
min,500.00,36.00,5.31,4.93,1.00,16.00,2007-08-01 00:00:00,-1.00,0.00,1934-04-01 00:00:00,1.00,0.00,0.00,2.00,0.00,0.00
25%,"8,000.00",36.00,9.75,249.01,3.00,"45,996.00",2014-06-01 00:00:00,11.80,0.00,1995-04-01 00:00:00,8.00,0.00,33.60,16.00,0.00,0.00
50%,"12,000.00",36.00,12.74,375.43,6.00,"65,000.00",2015-07-01 00:00:00,17.61,0.00,2000-07-01 00:00:00,11.00,0.00,52.30,23.00,0.00,0.00
75%,"20,000.00",36.00,15.99,580.56,10.00,"90,000.00",2016-06-01 00:00:00,24.04,0.00,2004-06-01 00:00:00,14.00,0.00,70.80,32.00,0.00,0.00
max,"40,000.00",60.00,30.99,"1,719.83",10.00,"10,999,200.00",2018-12-01 00:00:00,999.00,39.00,2015-09-01 00:00:00,90.00,86.00,892.30,176.00,12.00,1.00
std,"8,698.39",10.27,4.76,261.03,3.46,"69,968.46",NaN,10.94,0.88,NaN,5.46,0.60,24.50,12.00,0.38,0.40


In [55]:
pd.set_option('display.float_format', '{:,.2f}'.format)
df.describe()

,loan_amnt,term,int_rate,installment,emp_length,annual_inc,issue_d,dti,delinq_2yrs,earliest_cr_line,open_acc,pub_rec,revol_util,total_acc,pub_rec_bankruptcies,defaulted
count,"1,301,819.00","1,301,819.00","1,301,819.00","1,301,819.00","1,301,819.00","1,301,819.00",1301819,"1,301,819.00","1,301,819.00",1301819,"1,301,819.00","1,301,819.00","1,301,819.00","1,301,819.00","1,301,819.00","1,301,819.00"
mean,"14,420.23",41.80,13.26,438.18,6.05,"76,167.78",2015-05-21 06:53:31.065747,18.27,0.32,1999-02-17 02:53:22.692233,11.60,0.22,51.92,25.03,0.13,0.20
min,500.00,36.00,5.31,4.93,1.00,16.00,2007-08-01 00:00:00,-1.00,0.00,1934-04-01 00:00:00,1.00,0.00,0.00,2.00,0.00,0.00
25%,"8,000.00",36.00,9.75,249.01,3.00,"45,996.00",2014-06-01 00:00:00,11.80,0.00,1995-04-01 00:00:00,8.00,0.00,33.60,16.00,0.00,0.00
50%,"12,000.00",36.00,12.74,375.43,6.00,"65,000.00",2015-07-01 00:00:00,17.61,0.00,2000-07-01 00:00:00,11.00,0.00,52.30,23.00,0.00,0.00
75%,"20,000.00",36.00,15.99,580.56,10.00,"90,000.00",2016-06-01 00:00:00,24.04,0.00,2004-06-01 00:00:00,14.00,0.00,70.80,32.00,0.00,0.00
max,"40,000.00",60.00,30.99,"1,719.83",10.00,"10,999,200.00",2018-12-01 00:00:00,999.00,39.00,2015-09-01 00:00:00,90.00,86.00,892.30,176.00,12.00,1.00
std,"8,698.39",10.27,4.76,261.03,3.46,"69,968.46",NaN,10.94,0.88,NaN,5.46,0.60,24.50,12.00,0.38,0.40


In [56]:
print('dit = -1 :',(df['dti'] == -1).sum())
print('dti > 100:', (df['dti'] > 100).sum())

dit = -1 : 2
dti > 100: 476


In [57]:
before = len(df)
df = df[(df['dti'] >= 0)& (df['dti'] <= 100)]
after = len(df)
print('Dropped: ', before - after, 'rows')
print('Rows Left:', after)
print('dti range now: ', df['dti'].min(), "to", df['dti'].max())

Dropped:  478 rows
Rows Left: 1301341
dti range now:  0.0 to 100.0


Step 8 Duplicates 

In [58]:
print("Duplicate rows: ", df.duplicated().sum())

Duplicate rows:  0


Step 9 Saving the clean Dataset

In [59]:
df.to_csv('loan_clean.csv', index=False)
print('Saved! shape:', df.shape)

Saved! shape: (1301341, 23)


DOES LENDING CLUB'S OWN RISK GRADE ACTUALLY WORK??

In [60]:
df.groupby('grade')['defaulted'].mean().sort_index()

grade
A   0.06
B   0.13
C   0.23
D   0.30
E   0.39
F   0.45
G   0.50
Name: defaulted, dtype: float64

DOES THE BANK CHARGE MORE FOR THE RISK?

In [61]:
df.groupby('grade')['int_rate'].mean().sort_index()

grade
A    7.12
B   10.69
C   14.02
D   17.70
E   21.10
F   24.90
G   27.69
Name: int_rate, dtype: float64

In [62]:
df.groupby('term')['defaulted'].mean()

term
36   0.16
60   0.33
Name: defaulted, dtype: float64

In [63]:
df.groupby('purpose')['defaulted'].mean().sort_values(ascending=False)

purpose
small_business       0.30
renewable_energy     0.24
moving               0.23
medical              0.22
house                0.22
debt_consolidation   0.21
other                0.21
vacation             0.19
major_purchase       0.19
home_improvement     0.18
educational          0.17
credit_card          0.17
car                  0.15
wedding              0.12
Name: defaulted, dtype: float64

In [64]:
df.groupby('defaulted')[['int_rate', 'dti', 'annual_inc', 'emp_length']].mean()

,int_rate,dti,annual_inc,emp_length
defaulted,,,,
0,12.64,17.70,"77,654.09",6.08
1,15.71,20.06,"70,369.05",5.96


In [65]:
total_lent = df['loan_amnt'].sum()
defaulted_amnt = df[df['defaulted'] == 1]['loan_amnt'].sum()
pct_at_risk = defaulted_amnt/total_lent * 100

print(f'Total Lent: ${total_lent:,.0f}')
print(f'In defaulted loans: ${defaulted_amnt:,.0f}')
print(f'% of money at risk: {pct_at_risk:.1f}%')

Total Lent: $18,763,000,150
In defaulted loans: $4,061,573,400
% of money at risk: 21.6%
